# CEOAI / EUROAI Starter Sandbox Environment
## Task: Subgrid Mine-Risk Assessment

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score

BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Executing training workflow on device: {DEVICE}')

In [ ]:
class MinesweeperDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        grid = np.array([int(x) for x in row['board_state'].split()]).reshape(9, 9)
        
        mask = np.zeros((9, 9))
        mask[int(row['target_row']), int(row['target_col'])] = 1.0
        
        features = np.stack([grid, mask], axis=0).astype(np.float32)
        
        if self.is_test:
            return torch.tensor(features)
        else:
            label = int(row['is_mine'])
            return torch.tensor(features), torch.tensor(label, dtype=torch.long)

In [ ]:
class BoardEvaluatorCNN(nn.Module):
    def __init__(self):
        super(BoardEvaluatorCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels=2, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten()
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 9 * 9, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
        
    def forward(self, x):
        return self.classifier(self.conv_block(x))